In [ ]:
# Testing baru
import cv2
import mediapipe as mp
import numpy as np
import joblib
import torch
import torch.nn as nn
from collections import deque, Counter
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# DEFINISI ARSITEKTUR BI-GRU (Wajib sama dengan saat training)
# ==========================================================
class IsyaratBiGRU(nn.Module):
    def __init__(self, input_size=104, hidden_size=128, num_layers=2, num_classes=16):
        super(IsyaratBiGRU, self).__init__()
        self.bigru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        
    def forward(self, x):
        out, _ = self.bigru(x)
        out = self.fc(out[:, -1, :])
        return out

# ==========================================================
# LOAD MODEL
# ==========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_huruf = joblib.load("model_huruf_indo.pkl")
model_angka = joblib.load("model_angka.pkl")

# --- PERUBAHAN MODE KATA (MENGGUNAKAN PYTORCH BI-GRU) ---
model_kata = IsyaratBiGRU(input_size=104, hidden_size=128, num_layers=2, num_classes=16)
model_kata.load_state_dict(torch.load("model_gru_kata.pth", map_location=device))
model_kata.to(device)
model_kata.eval() # Set ke mode evaluasi

# Sesuaikan dengan urutan LABELS pada file training Anda (12 kata)
LABEL_KATA = {
    0: "berasal",
    1: "berpikir",
    2: "maaf",
    3: "makan",
    4: "mandi",
    5: "nama",
    6: "perkenalkan",
    7: "saya",
    8: "tidur",
    9: "tolong",
    10: "tunggu",
    11: "umur",
    12: "berteman",
    13: "sementara",
    14: "santai aja",
    15: "bareng",
}

# ==========================================================
# MEDIAPIPE
# ==========================================================
mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.6
)

holistic = mp_holistic.Holistic(
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

FACE_POINTS = [1, 33, 61, 199, 263]
POSE_POINTS = [11, 12, 13, 14, 15]

# ==========================================================
# STATE
# ==========================================================
mode = "huruf"

pred_buffer = deque(maxlen=8)
# --- BUFFER KHUSUS SEKUENS FRAME KATA (30 Frame) ---
frame_buffer = deque(maxlen=30) 

history_huruf = []
history_angka = []
history_kata = []

last_saved_huruf = ""
last_saved_angka = ""
last_saved_kata = ""

# ==========================================================
# HELPER
# ==========================================================
def extract_one_hand_live(hand_landmarks):
    lm = hand_landmarks.landmark
    pts = np.array([[p.x, p.y] for p in lm])
    base = pts[0]
    pts = pts - base
    max_val = np.max(np.abs(pts))
    if max_val != 0:
        pts = pts / max_val
    data = pts.flatten().tolist()

    def dist_live(a, b):
        return ((lm[a].x - lm[b].x) ** 2 + (lm[a].y - lm[b].y) ** 2) ** 0.5

    data.append(dist_live(4, 8))
    data.append(dist_live(8, 12))
    data.append(dist_live(12, 16))
    data.append(dist_live(16, 20))
    return data


def get_confidence(model, data):
    try:
        if hasattr(model, "predict_proba"):
            return float(np.max(model.predict_proba(data)[0]))
    except:
        return None
    return None


def get_stable_prediction(pred):
    pred_buffer.append(pred)
    if len(pred_buffer) < 6:
        return None
    kandidat, jumlah = Counter(pred_buffer).most_common(1)[0]
    if jumlah >= 5:
        return kandidat
    return None


def simpan_history(mode, pred):
    global last_saved_huruf, last_saved_angka, last_saved_kata
    if pred is None or pred == "-":
        return

    if mode == "huruf" and pred != last_saved_huruf:
        history_huruf.append(pred)
        last_saved_huruf = pred
    elif mode == "angka" and pred != last_saved_angka:
        history_angka.append(pred)
        last_saved_angka = pred
    elif mode == "kata" and pred != last_saved_kata:
        history_kata.append(pred)
        last_saved_kata = pred


def reset_buffer():
    pred_buffer.clear()
    frame_buffer.clear() # Ikut bersihkan buffer frame kata saat ganti mode


def draw_confidence_bar(frame, x, y, w, h, conf):
    cv2.rectangle(frame, (x, y), (x + w, y + h), (70, 70, 70), 1)
    if conf is not None:
        fill = int(w * conf)
        cv2.rectangle(frame, (x, y), (x + fill, y + h), (0, 220, 255), -1)


# ==========================================================
# CAMERA
# ==========================================================
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    pred = "-"
    stable_pred = None
    conf = None

    # ======================================================
    # MODE HURUF 
    # ======================================================
    if mode == "huruf":
        result_hands = hands_detector.process(rgb)
        if result_hands.multi_hand_landmarks:
            empty_hand = [0.0] * 46
            left_data = empty_hand.copy()
            right_data = empty_hand.copy()

            for hand_landmarks, handedness in zip(result_hands.multi_hand_landmarks, result_hands.multi_handedness):
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                hand_label = handedness.classification[0].label
                if hand_label == "Left":
                    left_data = extract_one_hand_live(hand_landmarks)
                elif hand_label == "Right":
                    right_data = extract_one_hand_live(hand_landmarks)

            data_huruf = left_data + right_data
            if len(data_huruf) == 92:
                data_huruf = np.array(data_huruf).reshape(1, -1)
                if isinstance(model_huruf, dict) and 'model' in model_huruf:
                    pred_encoded = model_huruf['model'].predict(data_huruf)[0]
                    pred = str(model_huruf['label_encoder'].inverse_transform([pred_encoded])[0])
                    conf = get_confidence(model_huruf['model'], data_huruf)
                else:
                    pred = str(model_huruf.predict(data_huruf)[0])
                    conf = get_confidence(model_huruf, data_huruf)

    # ======================================================
    # MODE ANGKA
    # ======================================================
    elif mode == "angka":
        result = holistic.process(rgb)
        hand = result.right_hand_landmarks or result.left_hand_landmarks
        if hand:
            mp_draw.draw_landmarks(frame, hand, mp_holistic.HAND_CONNECTIONS)
            lm_list = hand.landmark
            data = []
            base_x, base_y = lm_list[0].x, lm_list[0].y

            for lm in lm_list:
                data.append(lm.x - base_x)
                data.append(lm.y - base_y)

            data = np.array(data)
            data = data - np.min(data)
            if np.max(data) != 0: data = data / np.max(data)
            data = data.tolist()

            def dist(a, b): return ((a.x - b.x) ** 2 + (a.y - b.y) ** 2) ** 0.5
            data.append(dist(lm_list[4], lm_list[8]))
            data.append(dist(lm_list[8], lm_list[12]))
            data.append(dist(lm_list[12], lm_list[16]))
            data.append(dist(lm_list[16], lm_list[20]))

            if len(data) == 46:
                data = np.array(data).reshape(1, -1)
                pred = str(model_angka.predict(data)[0])
                conf = get_confidence(model_angka, data)

    # ======================================================
    # MODE KATA (LOGIKA ASLI + REM TANGAN TIDAK ADA)
    # ======================================================
    elif mode == "kata":
        result = holistic.process(rgb)
        
        # Gambar skeleton tangan jika terdeteksi
        if result.left_hand_landmarks: mp_draw.draw_landmarks(frame, result.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        if result.right_hand_landmarks: mp_draw.draw_landmarks(frame, result.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        # 1. CEK KEBERADAAN TANGAN (Kiri ATAU Kanan)
        tangan_terdeteksi = result.left_hand_landmarks is not None or result.right_hand_landmarks is not None

        feat_frame = []

        # Ekstrak 104 fitur dari frame saat ini
        if result.left_hand_landmarks:
            for lm in result.left_hand_landmarks.landmark: feat_frame += [lm.x, lm.y]
        else: feat_frame += [0.0] * 42

        if result.right_hand_landmarks:
            for lm in result.right_hand_landmarks.landmark: feat_frame += [lm.x, lm.y]
        else: feat_frame += [0.0] * 42

        if result.face_landmarks:
            for i in FACE_POINTS:
                lm = result.face_landmarks.landmark[i]
                feat_frame += [lm.x, lm.y]
        else: feat_frame += [0.0] * 10

        if result.pose_landmarks:
            for i in POSE_POINTS:
                lm = result.pose_landmarks.landmark[i]
                feat_frame += [lm.x, lm.y]
        else: feat_frame += [0.0] * 10

        # Masukkan 104 fitur frame ini ke antrean buffer sekuens
        frame_buffer.append(feat_frame)

        # 2. LOGIKA PREDIKSI DENGAN REM TANGAN
        # Prediksi hanya berjalan saat buffer sudah penuh DAN tangan wajib ada di kamera
        if len(frame_buffer) == 30 and tangan_terdeteksi:
            # Ubah data buffer menjadi tensor PyTorch dengan dimensi (1, 30, 104)
            tensor_input = torch.tensor([list(frame_buffer)], dtype=torch.float32).to(device)
            
            with torch.no_grad():
                outputs = model_kata(tensor_input)
                # Dapatkan nilai probabilitas menggunakan Softmax untuk hitung Confidence
                probabilities = torch.softmax(outputs, dim=1)
                conf_val, pred_idx_tensor = torch.max(probabilities, 1)
                
                pred_index = int(pred_idx_tensor.item())
                conf = float(conf_val.item())
                
                pred = LABEL_KATA.get(pred_index, f"?{pred_index}")
        else:
            # JIKA BUFFER BELUM 30 ATAU TANGAN TIDAK ADA: Langsung set ke "-"
            pred = "-"

    # ======================================================
    # STABILKAN PREDIKSI
    # ======================================================
    if pred != "-":
        stable_pred = get_stable_prediction(pred)
        simpan_history(mode, stable_pred)

    # ======================================================
    # UI BAR ATAS & BAWAH 
    # ======================================================
    h, w, _ = frame.shape
    TOP_H, BOT_H = 80, 35

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, TOP_H), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    overlay2 = frame.copy()
    cv2.rectangle(overlay2, (0, h - BOT_H), (w, h), (15, 15, 15), -1)
    cv2.addWeighted(overlay2, 0.55, frame, 0.45, 0, frame)

    cv2.putText(frame, "MODE", (15, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)
    cv2.putText(frame, mode.upper(), (15, 65), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 120), 2)

    pred_text = str(stable_pred if stable_pred else "-")
    text_size = cv2.getTextSize(pred_text, cv2.FONT_HERSHEY_SIMPLEX, 1.8, 3)[0]
    pred_x = (w - text_size[0]) // 2
    cv2.putText(frame, "PREDICTION", (pred_x, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)
    cv2.putText(frame, pred_text, (pred_x, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 220, 255), 3)

    conf_label_x = w - 210
    cv2.putText(frame, "CONFIDENCE", (conf_label_x, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)
    draw_confidence_bar(frame, conf_label_x, 32, 140, 14, conf)
    if conf is not None:
        cv2.putText(frame, f"{conf * 100:.1f}%", (conf_label_x, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, "A = Huruf   N = Angka   K = Kata   Q = Keluar", (20, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    cv2.imshow("Sign Language Detection", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("a"): mode = "huruf"; reset_buffer()
    elif key == ord("n"): mode = "angka"; reset_buffer()
    elif key == ord("k"): mode = "kata"; reset_buffer()
    elif key == ord("q"): break

cap.release()
cv2.destroyAllWindows()

# ==========================================================
# HISTORY CETAK
# ==========================================================
print("\n==============================")
print("HISTORY DETEKSI")
print("==============================")
for name, hist in [("HURUF", history_huruf), ("ANGKA", history_angka), ("KATA", history_kata)]:
    print(f"\n{name}:")
    if hist:
        for i, item in enumerate(hist, 1): print(f"{i}. {item}")
    else: print("-")